# Lesson 6 · Charged particles in electric and magnetic fields

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StativeSh/quantum-playground/blob/main/lessons/06_charged_particles_in_fields.ipynb)

**For:** A-level Physics (for example AQA 3.7.3.2 and 3.7.5.2, OCR A 6.2 and 6.3, Edexcel Topic 7) and IB Physics (D.3 motion in electromagnetic fields). Check your own specification for the exact wording.

**Before this lesson:** Lesson 2, *Projectile motion* (the parabola from a constant sideways force); Lesson 3, *Circular motion and gravitational orbits* (centripetal force); Lesson 5, *Electric fields and potential* (uniform fields between plates). Also assumed: Fleming's left-hand rule.

**Leads to:** Lesson 14, *Electromagnetic induction*.

**You will be able to:**

1. describe and calculate the parabolic path of a charge in a uniform electric field
2. use $F = BQv$ and derive $r = mv/BQ$ for circular motion in a magnetic field
3. explain why the period $T = 2\pi m/BQ$ does not depend on speed, and how a cyclotron uses this
4. explain how a velocity selector ($v = E/B$) and a mass spectrometer work
5. describe the helical path when the velocity is not perpendicular to the field

**How to use this lesson:** run each cell with Shift+Enter. Where you see *Try it*, change a number and run the cell again. Before some experiments a **Predict** box asks you to commit to an answer; write it down, run the cell, then answer the **Explain** box below it. Answers to both are at the very end, after the worked solutions. Exercises are at the end, with worked solutions below them.

In [ ]:
# Install the course API in a fresh Colab runtime or upgrade an older installation.
import importlib.util
import subprocess
import sys
from importlib.metadata import version

if importlib.util.find_spec("qplay") is None or int(version("quantum-playground").split(".")[0]) < 1:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "quantum-playground>=1,<2"], check=True)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from qplay import em

e = 1.602e-19         # C
m_e = 9.109e-31       # kg
m_p = 1.673e-27       # kg
u = 1.661e-27         # kg, atomic mass unit

## 1. A charge in a uniform electric field

Between parallel plates the field is uniform, so a charge feels a **constant** force $F = qE$, at right angles to the plates. An electron fired between the plates is just like a ball thrown horizontally:

- along the plates there is no force, so its velocity $v$ is constant and it takes time $t = x/v$ to travel a distance $x$
- across the plates it accelerates uniformly with $a = eE/m$

so its path is a **parabola**, $y = \dfrac{eE}{2mv^2}x^2$. Below the package integrates the force step by step and we compare.

In [ ]:
v0, E, L = 2.0e7, 1.0e4, 0.05          # m/s, V/m, plate length in m    <- Try it
# The electron is negative, so a field pointing down (-y) pushes it up (+y).
t, state = em.particle_trajectory(-e, m_e, E=[0, -E, 0], B=[0, 0, 0], r0=[0, 0, 0], v0=[v0, 0, 0], t_max=L / v0, n=200)
r, v = state[:, :3], state[:, 3:]

plt.figure(figsize=(7, 3.4))
plt.plot(r[:, 0] * 100, r[:, 1] * 1000, label="simulation")
xs = np.linspace(0, L, 15)
plt.plot(xs * 100, e * E * xs**2 / (2 * m_e * v0**2) * 1000, "o", ms=4, label=r"$y = eEx^2/2mv^2$")
plt.axhline(-1, color="0.5", lw=4); plt.axhline(8, color="0.5", lw=4)
plt.xlabel("x / cm"); plt.ylabel("y / mm"); plt.legend(loc="upper left"); plt.title("Electron between deflecting plates")
print(f"deflection on leaving the plates: {r[-1, 1] * 1000:.3f} mm; speed along the plates still {v[-1, 0]:.4e} m/s")

The horizontal velocity never changes; only the vertical one grows. This is how an old cathode-ray oscilloscope steered its beam.

## 2. A charge in a uniform magnetic field

A charge moving at right angles to a magnetic field feels a force

$$F = BQv$$

at right angles to **both** the field and the velocity (Fleming's left-hand rule, remembering that conventional current is opposite to electron flow). Because the force is always perpendicular to the velocity, it does no work: the speed never changes, only the direction. The result is a **circle**, with the magnetic force providing the centripetal force:

$$BQv = \frac{mv^2}{r}\quad\Rightarrow\quad r = \frac{mv}{BQ},\qquad T = \frac{2\pi r}{v} = \frac{2\pi m}{BQ}.$$

Look carefully at what the period formula depends on before you run the next cell.

**Predict (P1).** Three protons enter the field together at 0.5, 1.0 and $1.5\times10^6\ \mathrm{m\,s^{-1}}$. Which one gets back to its starting point first? Write your prediction down before you run the next cell.

In [ ]:
B = 0.10                                         # tesla, along +z
T_formula = 2 * np.pi * m_p / (e * B)
plt.figure(figsize=(6, 6))
for i, speed in enumerate([0.5e6, 1.0e6, 1.5e6]):
    t, state = em.particle_trajectory(e, m_p, E=[0, 0, 0], B=[0, 0, B], r0=[0, 0, 0], v0=[speed, 0, 0], t_max=T_formula, n=600)
    r, v = state[:, :3], state[:, 3:]
    radius = (r[:, 1].max() - r[:, 1].min()) / 2
    back_home = np.hypot(r[-1, 0], r[-1, 1])
    plt.plot(r[:, 0] * 100, r[:, 1] * 100, color=f"C{i}",
             label=f"v = {speed:.1e} m/s: r = {radius * 100:.2f} cm (mv/BQ = {m_p * speed / (e * B) * 100:.2f} cm)")
    print(f"v = {speed:.1e}: back within {back_home * 1e9:.2f} nm of the start after T = 2*pi*m/(BQ) = {T_formula * 1e9:.1f} ns; speed changed by {abs(np.linalg.norm(v[-1]) / speed - 1):.1e}")
plt.plot(0, 0, "ko")
plt.gca().set_aspect("equal"); plt.xlabel("x / cm"); plt.ylabel("y / cm")
plt.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.title("Protons in B = 0.10 T (field out of the page)");

**Explain (P1).** Why?

All three protons arrive back at the start at the same moment, after exactly $2\pi m/BQ$, and their speeds never change. The period does not depend on the speed: faster particles go round bigger circles in exactly the same time. That is the principle of the **cyclotron**.

## 3. Velocity selector

Put an electric field and a magnetic field at right angles to each other and to the beam. The electric force $QE$ pushes one way, the magnetic force $BQv$ the other. They balance only for one speed:

$$QE = BQv\quad\Rightarrow\quad v = \frac{E}{B}.$$

Only particles at exactly $E/B$ go straight through a slit at the far end. The charge and mass do not matter.

**Predict (P2).** A proton slower than $E/B$ enters the selector. Will it bend in the direction of the electric force or of the magnetic force? What about a faster one? Write your prediction down before you run the next cell.

In [ ]:
E, B, length = 4.0e4, 0.08, 0.10                 # V/m, T, m
v_pass = E / B
plt.figure(figsize=(7.5, 3.4))
for i, factor in enumerate([0.8, 0.95, 1.0, 1.05, 1.25]):
    speed = factor * v_pass
    # Positive ion moving along +x with B along +z: magnetic force is -y, electric force (E along +y) is +y.
    t, state = em.particle_trajectory(e, m_p, E=[0, E, 0], B=[0, 0, B], r0=[0, 0, 0], v0=[speed, 0, 0], t_max=1.5 * length / speed, n=450)
    r = state[:, :3]
    inside = r[:, 0] <= length                       # stop drawing at the slit
    plt.plot(r[inside, 0] * 100, r[inside, 1] * 1000, color=f"C{i}", label=f"{factor:.2f} E/B")
plt.plot([10, 10], [-30, -0.5], "k", lw=3); plt.plot([10, 10], [0.5, 30], "k", lw=3)      # exit slit
plt.xlabel("x / cm"); plt.ylabel("y / mm"); plt.ylim(-30, 30); plt.legend(fontsize=8, ncol=5, loc="lower left")
plt.title(f"Protons in a velocity selector: only v = E/B = {v_pass:.2e} m/s gets through the slit");

**Explain (P2).** Which of the two forces depends on speed, and how does that decide the answer?

## 4. Mass spectrometer

After the selector every ion has the same speed. They then enter a region with only a magnetic field and travel in semicircles of radius $r = mv/BQ$. For the same charge, **the radius is proportional to the mass**, so different isotopes land in different places. Here are the two main isotopes of neon, both singly ionised:

In [ ]:
v, B = 1.0e5, 0.50
plt.figure(figsize=(7, 4))
for i, (name, mass_u) in enumerate([("neon-20", 19.992), ("neon-22", 21.991)]):
    m = mass_u * u
    T = 2 * np.pi * m / (e * B)
    # Enters moving +y at the origin; B along +z (out of the page) bends a positive ion towards +x.
    t, state = em.particle_trajectory(e, m, E=[0, 0, 0], B=[0, 0, B], r0=[0, 0, 0], v0=[0, v, 0], t_max=T / 2, n=300)
    r = state[:, :3]
    plt.plot(r[:, 0] * 1000, r[:, 1] * 1000, color=f"C{i}")
    plt.plot(r[-1, 0] * 1000, 0, "v", color=f"C{i}", ms=10, label=f"{name}: lands at {r[-1, 0] * 1000:.2f} mm (2mv/BQ = {2 * m * v / (e * B) * 1000:.2f} mm)")
plt.axhline(0, color="k", lw=2)
plt.xlabel("x / mm (detector)"); plt.ylabel("y / mm"); plt.gca().set_aspect("equal"); plt.legend(fontsize=8, loc="upper left")
plt.title("Isotopes separate by mass");

## 5. When the velocity is not perpendicular to the field

Only the part of the velocity **perpendicular** to $B$ feels a force. The part **along** $B$ is unaffected. So the particle circles around the field line while drifting along it: a **helix**. This is how charged particles from the Sun spiral down the Earth's field lines to the poles and make the aurora.

The pitch (distance along $B$ per turn) is $p = v_\parallel T = v_\parallel\,\dfrac{2\pi m}{BQ}$.

In [ ]:
B = 0.10
t, state = em.particle_trajectory(e, m_p, E=[0, 0, 0], B=[0, 0, B], r0=[0, 0, 0], v0=[1e6, 0, 0.3e6], t_max=3 * 2 * np.pi * m_p / (e * B), n=900)
r = state[:, :3]
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(projection="3d")
ax.plot(r[:, 0] * 100, r[:, 1] * 100, r[:, 2] * 100)
ax.quiver(0, -10, 0, 0, 0, 15, color="C3", arrow_length_ratio=0.1)
ax.text(0, -10, 16, "B", color="C3")
ax.set_xlabel("x / cm"); ax.set_ylabel("y / cm"); ax.set_zlabel("z / cm"); ax.set_title("Proton spiralling along B");

---

## Exercises

Take $e = 1.60\times10^{-19}$ C, $m_e = 9.11\times10^{-31}$ kg, $m_p = 1.67\times10^{-27}$ kg and $1\ \mathrm{u} = 1.66\times10^{-27}$ kg. Give answers to 3 significant figures.

**Q1.** An electron moving at $3.0\times10^{7}\ \mathrm{m\,s^{-1}}$ enters a uniform magnetic field of 2.0 mT at right angles. Find the radius of its path.

**Q2.** Find the time the electron in Q1 takes to complete one circle.

**Q3.** A velocity selector has $E = 4.0\times10^{4}\ \mathrm{V\,m^{-1}}$ and $B = 0.080$ T. What speed passes through undeflected?

**Q4.** An electron travelling at $2.0\times10^{7}\ \mathrm{m\,s^{-1}}$ enters the space between two plates 5.0 cm long, moving parallel to them. The field between the plates is $1.0\times10^{4}\ \mathrm{V\,m^{-1}}$. How far has it been deflected sideways when it leaves the plates?

**Q5.** Singly charged ions of neon-20 (19.992 u) and neon-22 (21.991 u) enter a 0.50 T field at $1.0\times10^{5}\ \mathrm{m\,s^{-1}}$ and travel in semicircles. How far apart do they land?

**Q6 (simulation).** A proton enters a 0.10 T field with $1.0\times10^{6}\ \mathrm{m\,s^{-1}}$ perpendicular to the field and $1.0\times10^{6}\ \mathrm{m\,s^{-1}}$ along it. Find the pitch of its helix.

In [ ]:
def check(name, yours, correct, tol=0.01):
    if yours is None:
        print(f"{name}: not answered yet")
    elif abs(yours - correct) <= tol * abs(correct):
        print(f"{name}: correct")
    else:
        print(f"{name}: not yet, check your working (you gave {yours})")

# The correct values are computed here from the physics, not typed in.
_answers = {
    "Q1": m_e * 3.0e7 / (e * 2.0e-3),
    "Q2": 2 * np.pi * m_e / (e * 2.0e-3),
    "Q3": 4.0e4 / 0.080,
    "Q4": e * 1.0e4 * 0.05**2 / (2 * m_e * (2.0e7) ** 2),
    "Q5": 2 * 1.0e5 * (21.991 - 19.992) * u / (e * 0.50),
    "Q6": 1.0e6 * 2 * np.pi * m_p / (e * 0.10),
}

# Replace None with your answers, in SI units.
mine = {"Q1": None, "Q2": None, "Q3": None, "Q4": None, "Q5": None, "Q6": None}
for q in mine:
    check(q, mine[q], _answers[q])

---

## Worked solutions

Try the questions before reading on.

**Q1.** $r = \dfrac{mv}{BQ} = \dfrac{9.11\times10^{-31} \times 3.0\times10^7}{2.0\times10^{-3} \times 1.60\times10^{-19}} = 8.53\times10^{-2}\ \mathrm{m}$, about 8.5 cm.

**Q2.** $T = \dfrac{2\pi m}{BQ} = \dfrac{2\pi \times 9.11\times10^{-31}}{2.0\times10^{-3} \times 1.60\times10^{-19}} = 1.79\times10^{-8}\ \mathrm{s}$. The speed is not needed.

**Q3.** $v = E/B = 4.0\times10^4 / 0.080 = 5.00\times10^{5}\ \mathrm{m\,s^{-1}}$

**Q4.** Time between the plates $t = L/v = 0.05/2.0\times10^7 = 2.5\times10^{-9}$ s. Acceleration $a = eE/m = 1.76\times10^{15}\ \mathrm{m\,s^{-2}}$. Deflection $y = \tfrac12 at^2 = \tfrac12 \times 1.76\times10^{15} \times (2.5\times10^{-9})^2 = 5.50\times10^{-3}\ \mathrm{m}$, i.e. 5.50 mm.

**Q5.** Each lands a diameter $2r = 2mv/BQ$ from the entrance, so the separation is $\dfrac{2v\,\Delta m}{BQ} = \dfrac{2 \times 1.0\times10^5 \times 1.999 \times 1.66\times10^{-27}}{0.50 \times 1.60\times10^{-19}} = 8.29\times10^{-3}\ \mathrm{m}$, about 8.3 mm.

**Q6.** The period depends only on $m$, $B$ and $Q$: $T = 2\pi m/BQ = 6.56\times10^{-7}$ s. In that time the proton moves $p = v_\parallel T = 1.0\times10^6 \times 6.56\times10^{-7} = 0.656$ m along the field. The simulation below agrees.

In [ ]:
for q, value in _answers.items():
    print(f"{q}: {value:#.3g}")

_T = 2 * np.pi * m_p / (e * 0.10)
_t, _state = em.particle_trajectory(e, m_p, E=[0, 0, 0], B=[0, 0, 0.10], r0=[0, 0, 0], v0=[1e6, 0, 1e6], t_max=_T, n=400)
_r = _state[:, :3]
print(f"simulated: after one turn the proton is {_r[-1, 2]:.4f} m further along B, directly above where it started")

**Next:** in section 3, change the particle from a proton (`m_p`) to a neon-20 ion (`20 * u`). Does the selector still pass the same speed? Why are the wrong-speed ions bent less?

### Predict and explain: answers

**P1.** They all arrive together. A faster proton goes round a bigger circle ($r \propto v$), and the extra distance exactly cancels the extra speed: $T = 2\pi r/v = 2\pi m/BQ$, with no $v$ in it.

**P2.** Slower ones bend towards the electric force (upwards in the plot); faster ones towards the magnetic force (downwards). The electric force $QE$ is the same at every speed, but the magnetic force $BQv$ grows with speed: below $E/B$ it is the smaller of the two, above $E/B$ the larger.